# **SELF CHECK GPT (various implementations)** 

## **LLM PROMPTING WITH  OLLAMA**

```python

"""Score de divergence SelfCheckGPT LLM PROMPT."""

from berlue.core.schemas import Claim, SelfCheckScore
from berlue.llm.client import OllamaClient


def compute_divergence(
    claim: Claim,
    samples: list[str],
    client: OllamaClient | None = None
) -> SelfCheckScore:
    """Calcule le score de divergence d'une affirmation par rapport aux échantillons.
    On interroge le LLM pour savoir si chaque échantillon soutient l'affirmation.
    Plus les échantillons contredisent l'affirmation, plus le score de divergence est proche de 1.0.
    """
    if not samples:
        raise ValueError(
            f"❌ Impossible d'évaluer l'affirmation '{claim.id}' : "
            "la liste d'échantillons (samples) est vide. Le LLM a probablement échoué en amont."
        )

    client = client or OllamaClient()
    contradictions = 0

    # On compare notre affirmation atomique à chaque variation générée
    for sample in samples:
        # Prompt NLI (Natural Language Inference) "Zero-Shot"
        prompt = (
            f"Tu es un vérificateur de faits strict.\n\n"
            f"Contexte : {sample}\n"
            f"Affirmation : {claim.text}\n\n"
            "Question : Le contexte soutient-il ou confirme-t-il l'affirmation ? "
            "Réponds UNIQUEMENT par le mot OUI ou NON."
        )

        # Température à 0.0 : on veut une réponse 100% déterministe pour l'évaluation NLI
        response = client.generate(prompt=prompt, temperature=0.0).strip().upper()

        # Si la réponse ne commence pas par OUI, on considère que c'est une divergence
        # (Ça gère les "NON", mais aussi les "JE NE SAIS PAS" si le modèle est confus)
        if not response.startswith("OUI"):
            contradictions += 1

    # Calcul des scores
    divergence_score = contradictions / len(samples)
    confidence = 1.0 - divergence_score

    return SelfCheckScore(
        claim_id=claim.id,
        divergence_score=divergence_score,
        confidence=confidence
    )


```python
"""Score de divergence SelfCheckGPT (méthode NLI via Transformers)."""

from transformers import pipeline
from berlue.core.schemas import Claim, SelfCheckScore

# On utilise un "Singleton" (chargement paresseux) pour ne charger
# le modèle NLI en mémoire qu'à la première utilisation.
_NLI_PIPELINE = None

def get_nli_pipeline():
    global _NLI_PIPELINE
    if _NLI_PIPELINE is None:
        print("⏳ Chargement du modèle NLI (DeBERTa) en mémoire...")
        # cross-encoder/nli-deberta-v3-small est le standard léger et rapide pour ça.
        _NLI_PIPELINE = pipeline(
            "text-classification",
            model="cross-encoder/nli-deberta-v3-small",
            device=-1  # -1 = CPU. Mettre 0 si tu as un GPU Nvidia disponible.
        )
    return _NLI_PIPELINE


def compute_divergence(claim: Claim, samples: list[str]) -> SelfCheckScore:
    """Calcule le score de divergence d'une affirmation par rapport aux échantillons
    en utilisant un modèle NLI dédié (très rapide).
    """
    if not samples:
        raise ValueError(
            f"❌ Impossible d'évaluer l'affirmation '{claim.id}' : "
            "la liste d'échantillons (samples) est vide. Le LLM a probablement échoué en amont."
        )

    nli_pipe = get_nli_pipeline()

    # On prépare les paires (Échantillon, Affirmation)
    # Le modèle va vérifier si l'échantillon implique l'affirmation
    pairs = [{"text": sample, "text_pair": claim.text} for sample in samples]

    # Inférence par lot (batch) : ultra rapide !
    results = nli_pipe(pairs)

    # Calcul du score d'incohérence selon la méthode SelfCheckGPT
    # On cherche la probabilité du label "contradiction"
    contradiction_probs = []

    for result in results:
        # Le modèle renvoie un dict ex: {'label': 'contradiction', 'score': 0.85}
        # Attention : selon les modèles, les labels peuvent s'appeler 'LABEL_2' ou 'contradiction'
        label = result["label"].lower()
        score = result["score"]

        if label == "contradiction":
            contradiction_probs.append(score)
        else:
            # Si le modèle a prédit 'entailment' (soutenu) ou 'neutral',
            # la probabilité de contradiction est considérée comme très faible.
            # (Une implémentation plus stricte récupère tous les logits, mais on simplifie ici)
            contradiction_probs.append(1.0 - score if label == "entailment" else 0.5)

    # La divergence est la moyenne des probabilités de contradiction
    divergence_score = sum(contradiction_probs) / len(contradiction_probs)

    return SelfCheckScore(
        claim_id=claim.id,
        divergence_score=divergence_score,
        confidence=1.0 - divergence_score
    )
    

## **[SelfCheckNLI](https://github.com/potsawee/selfcheckgpt)**

```python

"""Score de divergence SelfCheckGPT par affirmation (méthode NLI officielle)."""

import torch
from selfcheckgpt.modeling_selfcheck import SelfCheckNLI

from berlue.core.schemas import Claim, SelfCheckScore

# Variable globale pour garder le modèle en mémoire (Singleton)
_SELFCHECK_NLI_MODEL = None

def get_selfcheck_nli() -> SelfCheckNLI:
    """Charge le modèle NLI une seule fois en mémoire (sur GPU si disponible)."""

    global _SELFCHECK_NLI_MODEL
    if _SELFCHECK_NLI_MODEL is None:
        # Détection automatique du matériel (Nvidia CUDA ou CPU)
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"⏳ Initialisation de SelfCheckNLI sur : {device}...")
        _SELFCHECK_NLI_MODEL = SelfCheckNLI(device=device)
    return _SELFCHECK_NLI_MODEL


def compute_divergence(claim: Claim, samples: list[str]) -> SelfCheckScore:
    """Calcule le score de divergence d'une affirmation par rapport aux échantillons
    en utilisant le package officiel SelfCheckGPT (modèle NLI).
    """
    if not samples:
        raise ValueError(
            f"❌ Impossible d'évaluer l'affirmation '{claim.id}' : "
            "la liste d'échantillons (samples) est vide. Le LLM a probablement échoué en amont."
        )

    # Récupération du modèle (déjà chargé en VRAM/RAM)
    selfcheck_nli = get_selfcheck_nli()

    # Le package attend une LISTE de phrases. On met notre 'claim.text' dans une liste.
    # Il va comparer cette phrase avec la liste des 'samples'.
    scores = selfcheck_nli.predict(
        sentences=[claim.text], 
        sampled_passages=samples
    )
    
    # scores est un tableau numpy (ex: [0.334014]), on extrait la première valeur
    divergence = float(scores[0])

    return SelfCheckScore(
        claim_id=claim.id,
        divergence_score=divergence,
        confidence=1.0 - divergence
    )


## **SelfCheckNLI USAGE**

## 

In [1]:
from berlue.core.schemas import Claim

from berlue.selfcheck.scorer import compute_divergence

def test_sur_laptop():
    print("🚀 Début du test...")

    # On simule une affirmation extraite
    claim = Claim(
        id="claim_123",
        text="Michael Jordan a remporté 6 championnats NBA.",
        source_answer="Le joueur de basket Michael Jordan a remporté 6 championnats NBA avec les Bulls."
    )

    # 2. On simule les échantillons générés par le LLM Ollama

    samples = [
        "Michael Jordan est célèbre pour ses 6 bagues de champion NBA.", # Soutient
        "Il a gagné 6 titres NBA durant sa carrière.", # Soutient
        "Michael Jordan n'a gagné que 2 championnats dans sa vie." # Contredit
    ]

    print("⏳ Calcul du score (le premier lancement va télécharger ~1.5 Go de modèle)...")

    # 3. Calcul du score via SelfCheckNLI
    score = compute_divergence(claim=claim, samples=samples)

    print("\n✅ Résultat :")
    print(f"ID : {score.claim_id}")
    print(f"Divergence (0.0 = parfait, 1.0 = hallucination totale) : {score.divergence_score:.2f}")
    print(f"Confiance : {score.confidence:.2f}")


/home/justvnr/.pyenv/versions/berlue-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# POur télécharger la grosse berte
# A exécuter dans un terminal pour voir l'avancement...
# !hf download potsawee/deberta-v3-large-mnli

In [2]:
test_sur_laptop()

🚀 Début du test...
⏳ Calcul du score (le premier lancement va télécharger ~1.5 Go de modèle)...
⏳ Initialisation de SelfCheckNLI sur : cpu...
SelfCheck-NLI initialized to device cpu

✅ Résultat :
ID : claim_123
Divergence (0.0 = parfait, 1.0 = hallucination totale) : 0.34
Confiance : 0.66
